# ADAS simulation: real-time vehicle detection with collision warning
This notebook is the **demo / simulation** for the project. It runs the detectors on a road video (or KITTI frames), draws a dashcam-style ADAS display and saves an annotated video for each model:

- detection boxes with labels and confidence
- a live **FPS counter** and vehicle count
- an **ego-lane zone** and a **forward collision warning** (green → orange → red) when a vehicle or pedestrian in your lane gets close
- an **edge-device simulation**: the same model is run on the GPU and then on a **limited-thread CPU**, to show what constrained hardware does to FPS

**Works right now:** if you haven't trained anything yet, it uses the pretrained COCO weights so you can demo today. Once the main notebook has produced `adas_benchmark/weights/yolov8n.pt` and `yolov9t.pt` in your Drive, this notebook picks up your KITTI-trained weights automatically.

**Honest note:** the warning logic is a simple visual heuristic (box height in the ego lane), not a calibrated distance measurement. Say so in your presentation.

Run on a T4 GPU runtime: `Runtime > Run all`.

## 1. Setup

In [ ]:
!pip -q install ultralytics pandas
import os, time, shutil, base64, random
from pathlib import Path
import cv2, numpy as np, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU (the edge-CPU simulation still works)")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/adas_benchmark")
except Exception as e:
    print("Drive not mounted:", e)
    OUT = Path("/content/adas_benchmark")
WTS, FIG, SIM = OUT / "weights", OUT / "figures", OUT / "simulation"
for p in (WTS, FIG, SIM):
    p.mkdir(parents=True, exist_ok=True)

## 2. Settings

In [ ]:
VIDEO_PATH = None        # e.g. "/content/road.mp4" after uploading a dashcam clip (see next cell). None = use KITTI frames.
MAX_FRAMES = 120         # frames per run (more = longer)
FRAME_W = 960            # output width
CONF = 0.35              # detection confidence shown on screen
IMGSZ = 640
EDGE_THREADS = 2         # CPU threads for the edge-device simulation
MODELS = {"YOLOv8n": "yolov8n", "YOLOv9-tiny": "yolov9t"}

import torch
MODES = ([("GPU (T4)", 0, None)] if torch.cuda.is_available() else []) + [(f"Edge sim: CPU {EDGE_THREADS} threads", "cpu", EDGE_THREADS)]
def weights_for(tag):
    f = WTS / f"{tag}.pt"
    if f.exists():
        return str(f), "KITTI fine-tuned"
    return f"{tag}.pt", "COCO pretrained (train the main notebook for KITTI weights)"
for name, tag in MODELS.items():
    print(name, "->", weights_for(tag))

### Optional: upload your own dashcam video (the demo looks best with a real driving clip)

In [ ]:
# Run this cell only if you want to use your own video. Choose an .mp4 of a road drive.
from google.colab import files
up = files.upload()
VIDEO_PATH = "/content/" + next(iter(up))
print("Using video:", VIDEO_PATH)

## 3. ADAS display logic (lane zone, proximity levels, HUD)

In [ ]:
import cv2, numpy as np

VEHICLES = {"car", "van", "truck", "tram", "bus", "motorcycle", "bicycle", "cyclist"}
PEOPLE = {"pedestrian", "person", "person_sitting"}
CAUTION_H, WARNING_H = 0.20, 0.32     # box height / frame height. Heuristic proximity levels, no camera calibration.
GREEN, ORANGE, RED = (80, 200, 80), (0, 165, 255), (40, 40, 230)

def lane_polygon(W, H):
    """Ego-lane zone in front of the car (a trapezoid on the road ahead)."""
    return np.array([[0.30 * W, H], [0.70 * W, H], [0.56 * W, 0.55 * H], [0.44 * W, 0.55 * H]], np.int32)

def assess(boxes, labels, names, W, H):
    """Returns overall level (0 safe, 1 caution, 2 warning) and a status per box."""
    poly, level, stats = lane_polygon(W, H), 0, []
    for (x1, y1, x2, y2), c in zip(boxes, labels):
        st = 0
        if names[int(c)] in VEHICLES | PEOPLE and cv2.pointPolygonTest(poly, (float((x1 + x2) / 2), float(y2)), False) >= 0:
            rel = (y2 - y1) / H
            st = 2 if rel >= WARNING_H else 1 if rel >= CAUTION_H else 0
        stats.append(st)
        level = max(level, st)
    return level, stats

def adas_overlay(frame, boxes, scores, labels, names, fps, model_name, device_label):
    out = frame.copy()
    H, W = out.shape[:2]
    level, stats = assess(boxes, labels, names, W, H)
    lane = out.copy()
    cv2.fillPoly(lane, [lane_polygon(W, H)], (200, 160, 60))
    out = cv2.addWeighted(lane, 0.18, out, 0.82, 0)
    n_veh = 0
    for (x1, y1, x2, y2), s, c, st in zip(boxes, scores, labels, stats):
        n = names[int(c)]
        n_veh += n in VEHICLES
        col = (GREEN, ORANGE, RED)[st]
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), col, 2)
        cv2.putText(out, f"{n} {s:.2f}", (int(x1), max(14, int(y1) - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1, cv2.LINE_AA)
    # HUD panel
    cv2.rectangle(out, (8, 8), (300, 92), (20, 20, 20), -1)
    for i, txt in enumerate([f"Model: {model_name}", f"Device: {device_label}", f"FPS: {fps:5.1f}   Vehicles: {n_veh}"]):
        cv2.putText(out, txt, (16, 30 + i * 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
    if level:
        msg, col = (("CAUTION: OBJECT AHEAD", ORANGE), ("FORWARD COLLISION WARNING", RED))[level - 1]
        (tw, th), _ = cv2.getTextSize(msg, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
        x0 = max((W - tw) // 2, 330)
        cv2.rectangle(out, (x0 - 14, 10), (x0 + tw + 14, 20 + th + 10), col, -1)
        cv2.putText(out, msg, (x0, 20 + th), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    return out, level

## 4. Frame source

In [ ]:
def frame_source():
    """Yields frames resized to FRAME_W. Uses your video if set, else KITTI validation images."""
    def fit(f):
        h, w = f.shape[:2]
        return cv2.resize(f, (FRAME_W, int(h * FRAME_W / w)))
    if VIDEO_PATH:
        cap = cv2.VideoCapture(VIDEO_PATH)
        n = 0
        while n < MAX_FRAMES:
            ok, f = cap.read()
            if not ok:
                break
            yield fit(f); n += 1
        cap.release()
    else:
        from ultralytics.data.utils import check_det_dataset
        d = check_det_dataset("kitti.yaml")
        root = Path(d["val"] if isinstance(d["val"], str) else d["val"][0])
        imgs = sorted(str(p) for p in root.rglob("*") if p.suffix.lower() in (".png", ".jpg"))[:MAX_FRAMES]
        for p in imgs:
            yield fit(cv2.imread(p))

src_fps = 10 if not VIDEO_PATH else (cv2.VideoCapture(VIDEO_PATH).get(cv2.CAP_PROP_FPS) or 25)
print("Source:", VIDEO_PATH or "KITTI validation frames (not a continuous drive; upload a dashcam clip for a smoother demo)")

## 5. Run the simulation (one video per model and device mode)

In [ ]:
from ultralytics import YOLO
import pandas as pd

def run_sim(model_name, weights, wtype, mode_label, device, threads):
    if threads:
        torch.set_num_threads(threads)
    model = YOLO(weights)
    names = model.names
    out_raw = SIM / f"raw_{model_name}_{'gpu' if device == 0 else 'edge'}.mp4"
    writer, times, veh, ema, first_warn = None, [], [], 0.0, None
    levels = []
    for i, frame in enumerate(frame_source()):
        t0 = time.perf_counter()
        r = model.predict(frame, imgsz=IMGSZ, conf=CONF, device=device, verbose=False)[0]
        if device == 0:
            torch.cuda.synchronize()
        dt = time.perf_counter() - t0
        if i >= 3:                                   # skip warm-up frames
            times.append(dt)
            ema = (1 / dt) if ema == 0 else 0.9 * ema + 0.1 * (1 / dt)
        b = r.boxes
        shown, level = adas_overlay(frame, b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int),
                                    names, ema, f"{model_name} ({wtype.split(' (')[0]})", mode_label)
        levels.append(level)
        if level == 2 and first_warn is None:
            first_warn = shown.copy()
        veh.append(sum(names[int(c)] in VEHICLES for c in b.cls.cpu().numpy()))
        if writer is None:
            writer = cv2.VideoWriter(str(out_raw), cv2.VideoWriter_fourcc(*"mp4v"), src_fps, (shown.shape[1], shown.shape[0]))
        writer.write(shown)
    writer.release()
    final = str(out_raw).replace("raw_", "sim_")
    os.system(f'ffmpeg -y -loglevel error -i "{out_raw}" -vcodec libx264 -pix_fmt yuv420p "{final}"')   # browser-playable
    os.remove(out_raw)
    t = np.array(times)
    return {"model": model_name, "mode": mode_label, "weights": wtype, "frames": len(levels),
            "avg_latency_ms": round(t.mean() * 1000, 1), "avg_fps": round(1 / t.mean(), 1),
            "avg_vehicles_per_frame": round(float(np.mean(veh)), 1),
            "warning_frames": int(sum(l == 2 for l in levels)), "video": final}, first_warn

rows, warn_imgs = [], []
for mname, tag in MODELS.items():
    w, wtype = weights_for(tag)
    for mode_label, device, threads in MODES:
        r, fw = run_sim(mname, w, wtype, mode_label, device, threads)
        rows.append(r)
        if fw is not None:
            warn_imgs.append((f"{mname}, {mode_label}", fw))
        print(r["model"], "|", r["mode"], "|", r["avg_fps"], "FPS")
res = pd.DataFrame(rows)
res.drop(columns="video").to_csv(SIM / "simulation_summary.csv", index=False)
display(res.drop(columns="video"))

## 6. Watch the simulations

In [ ]:
from IPython.display import HTML, display
def show_video(path, width=800):
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<p><b>{Path(path).name}</b></p><video width="{width}" controls loop><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))
for _, r in res.iterrows():
    if "Edge" in r["mode"]:
        show_video(r["video"])
print("All videos are saved in:", SIM)

## 7. Images and charts for the slides

In [ ]:
import matplotlib.pyplot as plt
if warn_imgs:
    pick = warn_imgs[:4]
    fig, axes = plt.subplots(1, len(pick), figsize=(7 * len(pick), 3.6), squeeze=False)
    for ax, (title, im) in zip(axes[0], pick):
        ax.imshow(im[..., ::-1]); ax.axis("off"); ax.set_title(title)
    plt.tight_layout(); plt.savefig(FIG / "adas_collision_warning_frames.png", dpi=130); plt.show()
else:
    print("No collision-warning frame occurred in this clip. Try a clip with closer traffic, or lower WARNING_H in section 3.")

pv = res.pivot(index="model", columns="mode", values="avg_fps")
ax = pv.plot(kind="bar", figsize=(8, 4.5), rot=0)
ax.set_ylabel("Average FPS"); ax.set_title("Same model, different hardware (edge-device simulation)")
ax.axhline(30, color="gray", ls="--", lw=1); ax.text(0.02, 31, "30 FPS real-time line", transform=ax.get_yaxis_transform(), fontsize=9, color="gray")
plt.tight_layout(); plt.savefig(FIG / "edge_simulation_fps.png", dpi=150); plt.show()

In [ ]:
zip_path = shutil.make_archive("/content/adas_simulation", "zip", str(SIM))
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Everything is in:", SIM)

## Talking points for the demo
- "The detector runs frame by frame; the HUD shows live FPS and how many vehicles it sees."
- "The shaded zone is our ego lane. If a vehicle or pedestrian in that zone gets large in the frame, the system goes orange, then red with a collision warning."
- "Running the same model on a 2-thread CPU shows what happens on weaker edge hardware. That is the accuracy / speed / size trade-off our benchmark measures."
- Limitation to mention: proximity comes from box size, not a calibrated distance, and the Colab CPU is much faster than a real Raspberry Pi or Jetson Nano.